<a href="https://www.kaggle.com/code/augustinekuo/mlb-analytics-predictive-eda-and-regression?scriptVersionId=345018563" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# MLB Analytics & Predictive Modeling — v2 (Position-Aware Multiple Model Directory)

**Author**: Austin Kuo
**Target System**: Panel/Sports Analytics Suite in this repo (`app.py` → `macroservice/` + `utils/`)

---

| Group | Positions | Stat group (per `utils/filters.py`) |
|-------|-----------|--------------------------------------|
| **Battery** | P, C | P → pitching; C → hitting |
| **Infield** | 1B, 2B, 3B, SS | hitting |
| **Outfield** | LF, CF, RF | hitting |
| **Non-Fielders** | DH, TWP, PH, PR, UTL | hitting |

This notebook re-does the original's ingredient recipe (game-log ingestion,
rolling aggregates, widget-free regression) but:
1. Routes every representative player through the same position-group taxonomy
   the dashboard uses.
2. Replaces the single 80/20 chronological holdout with a **walk-forward
   `TimeSeriesSplit`** evaluation (no look-ahead leakage), keeping the repo's
   holdout split for the final trajectory charts.
3. Swaps the fixed `SVR/Huber/GP` blend for a **model zoo + `GridSearchCV`** pick
   so the "better fit on the data" is chosen by cross-validated score, not by
   hard-coded constants.

---

## Notebook flow
1. **Environment** — constants, safe casters, URL / header config.
2. **Position taxonomy** — mirror of `utils/positions.py` + `utils/filters.py`.
3. **Team roster** — tag every player with group + stat-group; pick one
   representative per group.
4. **Metric registry** — hitting vs pitching metric keys (`utils/filters.py`).
5. **Ingestion** — multi-season game logs + Statcast.
6. **Aggregation** — wOBA (offensive), FIP (pitching), rolling targets.
7. **Validation harness** — `TimeSeriesSplit` walk-forward + gauge table.
8. **Model zoo** — Ridge baseline vs SVR/Huber/GP/RF/HistGB/ensemble.
9. **Hyperparameter tuning** — `GridSearchCV` over a `TimeSeriesSplit`.
10. **Best-model trajectories** — 95% CI band charts per group.
11. **Summary** + live-news tail (inherited from the original notebook).


In [1]:
import io, os, warnings
import functools
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import xml.etree.ElementTree as ET

warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, VotingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE_URL = "https://statsapi.mlb.com/api/v1"
HEADERS = {"User-Agent": "Mozilla/5.0 MLB-Analytics-Dashboard-Kaggle-v2.1 (Austin Kuo)"}
SEARCH_URL = "https://baseballsavant.mlb.com/statcast_search/csv"

# ---- Speed controls -------------------------------------------------
# FAST_MODE=True skips the live network entirely and uses the synthetic
# generator for every player. This removes the single biggest source of
# slowness (multi-season HTTP round trips per player) while keeping the
# exact same downstream modeling code path.
FAST_MODE = True
USE_LIVE_DATA = not FAST_MODE
SEASONS_FAST = [2023, 2024]          # only used if USE_LIVE_DATA
SEASONS_FULL = [2018, 2019, 2020, 2021, 2022]
REQUEST_TIMEOUT = 8                   # was 20s; fail fast instead of hanging

def safe_float(val, default=np.nan):
    try:
        s = str(val).strip()
        return default if s in ("", "-", "--", "---", ".", "INF", "inf") else float(s)
    except (ValueError, TypeError):
        return default

def safe_int(val, default=0):
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return default

print("Setup complete. FAST_MODE =", FAST_MODE)


Setup complete. FAST_MODE = True


# 2. Position Taxonomy (mirrors `utils/positions.py`)

The dashboard never uses an "Offense vs Defense" dichotomy. It classifies every
rostered player into one of four position groups and, within those, assigns a
stat group from the position acronym:

- **Battery** {P, C} → P uses pitching metrics; C uses hitting metrics
- **Infield** {1B, 2B, 3B, SS} → hitting metrics
- **Outfield** {LF, CF, RF} → hitting metrics
- **Non-Fielders** {DH, TWP, PH, PR, UTL} → hitting metrics


In [2]:

# Literal copy of the shipped taxonomy in utils/positions.py so this notebook
# and the dashboard can never drift apart.
POSITION_GROUPS = {
    'Battery': ['P', 'C'],
    'Infield': ['1B', '2B', '3B', 'SS'],
    'Outfield': ['LF', 'CF', 'RF'],
    'Non-Fielders': ['DH', 'TWP', 'PH', 'PR', 'UTL'],
}

GROUP_FOR_POSITION = {
    position: group
    for group, positions in POSITION_GROUPS.items()
    for position in positions
}

def stat_group_for_position(position_abbr):
    """Mirror of utils/filters.py: only 'P' gets pitching metrics."""
    return 'pitching' if position_abbr == 'P' else 'hitting'

print('Groups:', list(POSITION_GROUPS))
print('Samples:', {p: (GROUP_FOR_POSITION[p], stat_group_for_position(p))
                   for p in ['C', 'P', '1B', 'LF', 'RF', 'DH']})


Groups: ['Battery', 'Infield', 'Outfield', 'Non-Fielders']
Samples: {'C': ('Battery', 'hitting'), 'P': ('Battery', 'pitching'), '1B': ('Infield', 'hitting'), 'LF': ('Outfield', 'hitting'), 'RF': ('Outfield', 'hitting'), 'DH': ('Non-Fielders', 'hitting')}


# 3. Team Roster & One Representative Per Group

The dashboard resolves a team's roster and each player's position acronym, then
runs them through the position taxonomy above. This notebook does the same for
one target team and then hand-picks **one well-established representative per
group** so every modeling cell below exercises a different position/stat-group
combination instead of just "baggage offense vs pitching defense".


In [3]:

# One representative per group. Multi-season windows give walk-forward CV enough
# folds. Player ids are best-effort from the live roster lookup below; when a
# season comes back empty (retired/traded/off-year, or no network on Kaggle),
# the synthetic-loader later substitutes a realistic demo series so the whole
# modeling pipeline still runs end-to-end.
REPRESENTATIVES = [
    ('Battery', 'P',  'pitching', 'David Bednar',    670280),
    ('Battery', 'C',  'hitting',  'Austin Wells',    None),
    ('Infield', '2B', 'hitting',  'Jazz Chisholm',   665862),
    ('Outfield', 'LF','hitting',  'Cody Bellinger',  641355),
    ('Non-Fielders','DH','hitting','Giancarlo Stanton', 519317),
]

for rep in REPRESENTATIVES:
    grp, pos, sgrp, name, pid = rep
    print(f"{grp:>13} | {pos:>2} -> {sgrp:>8} | {name} (id={pid})")
print('Representative matrix sized', len(REPRESENTATIVES))


      Battery |  P -> pitching | David Bednar (id=670280)
      Battery |  C ->  hitting | Austin Wells (id=None)
      Infield | 2B ->  hitting | Jazz Chisholm (id=665862)
     Outfield | LF ->  hitting | Cody Bellinger (id=641355)
 Non-Fielders | DH ->  hitting | Giancarlo Stanton (id=519317)
Representative matrix sized 5


# 4. Metric Registry (mirrors `utils/filters.py`)

Each stat-group has its own metric set. Rate stats live on a 0-to-low-single
digit scale (`RATE_METRICS`) while counting stats run to the hundreds; the
dashboard plots them on separate axes. The notebook uses the **rate** targets for
regression (rolling smoothed), the same convention as the threshold chart.


In [4]:

HITTING_METRICS = [
    ('avg', 'AVG'), ('obp', 'OBP'), ('slg', 'SLG'), ('ops', 'OPS'),
    ('homeRuns', 'HR'), ('rbi', 'RBI'), ('strikeOuts', 'K'), ('baseOnBalls', 'BB'),
    ('xba', 'xBA'), ('avgExitVelocity', 'EV'), ('hardHitPct', 'Hard-Hit%'), ('barrelPct', 'Barrel%'),
]
PITCHING_METRICS = [
    ('era', 'ERA'), ('whip', 'WHIP'), ('strikeOuts', 'K'), ('baseOnBalls', 'BB'),
    ('inningsPitched', 'IP'), ('earnedRuns', 'ER'),
    ('cswPct', 'CSW%'), ('whiffPct', 'Whiff%'), ('chasePct', 'Chase%'), ('avgVelocity', 'Velo'),
]
RATE_METRICS = frozenset({'avg', 'obp', 'slg', 'ops', 'era', 'whip',
                          'xba', 'hardHitPct', 'barrelPct', 'cswPct', 'whiffPct', 'chasePct'})

def metrics_for_group(group):
    return PITCHING_METRICS if group == 'pitching' else HITTING_METRICS

print('Hitting targets:', [a for a, _ in HITTING_METRICS])
print('Pitching targets:', [a for a, _ in PITCHING_METRICS])


Hitting targets: ['avg', 'obp', 'slg', 'ops', 'homeRuns', 'rbi', 'strikeOuts', 'baseOnBalls', 'xba', 'avgExitVelocity', 'hardHitPct', 'barrelPct']
Pitching targets: ['era', 'whip', 'strikeOuts', 'baseOnBalls', 'inningsPitched', 'earnedRuns', 'cswPct', 'whiffPct', 'chasePct', 'avgVelocity']


# 5. Ingestion → Aggregation → Features

Merges the original notebook's recipe with the repo's `macroservice/features.py`
rolling conventions:

- **Batter game log** → per-game hits/BB/HR etc. + `rolling_metric` per `metric`.
- **Pitcher game log** → per-appearance ERA/WHIP + rolling FIP (recomputed here).
- **wOBA / FIP** composite aggregates from the original notebook.
- **Feature columns** mirror `HITTER_FEATURE_COLUMNS`/`PITCHER_FEATURE_COLUMNS`:
  `appearance_num`, `momentum_3`, `is_home`, `rest_days`, plus rolling version of
  the target itself.


In [5]:
# ---- MLB Stats API game-log ingestion (per player, per season) -------
# Cached so re-running any downstream cell never re-hits the network for
# the same player/season pair. This alone removes most repeat-run lag.

@functools.lru_cache(maxsize=256)
def _fetch_gl_cached(player_id, season, group):
    r = requests.get(
        f"{BASE_URL}/people/{player_id}/stats",
        params={"stats": "gameLog", "group": group, "season": season, "gameType": "R"},
        headers=HEADERS, timeout=REQUEST_TIMEOUT,
    )
    return tuple(r.json().get("stats", [{}])[0].get("splits", []))

def fetch_batter_gl(player_id, seasons):
    frames = []
    for yr in sorted(seasons):
        try:
            splits = _fetch_gl_cached(player_id, yr, "hitting")
        except Exception:
            splits = ()
        for s in splits:
            st = s.get("stat", {})
            h, d, t, hr = (safe_int(st.get(k)) for k in ("hits", "doubles", "triples", "homeRuns"))
            frames.append(dict(
                season=yr, date=s.get("date"), is_home=s.get("isHome", False),
                AB=safe_int(st.get("atBats")), BB=safe_int(st.get("baseOnBalls")),
                SF=safe_int(st.get("sacFlies")), HBP=safe_int(st.get("hitByPitch")),
                K=safe_int(st.get("strikeOuts")),
                s1B=h - d - t - hr, s2B=d, s3B=t, HR=hr,
                avg=safe_float(st.get("avg")), obp=safe_float(st.get("obp")),
                slg=safe_float(st.get("slg")), ops=safe_float(st.get("ops")),
            ))
    return pd.DataFrame(frames)

def fetch_pitcher_gl(player_id, seasons):
    frames = []
    for yr in sorted(seasons):
        try:
            splits = _fetch_gl_cached(player_id, yr, "pitching")
        except Exception:
            splits = ()
        for s in splits:
            st = s.get("stat", {})
            frames.append(dict(
                season=yr, date=s.get("date"), is_home=s.get("isHome", False),
                IP=safe_float(st.get("inningsPitched"), 0.0), HR=safe_int(st.get("homeRuns")),
                BB=safe_int(st.get("baseOnBalls")), HBP=safe_int(st.get("hitByPitch")),
                K=safe_int(st.get("strikeOuts")), ER=safe_int(st.get("earnedRuns")),
                era=safe_float(st.get("era")), whip=safe_float(st.get("whip")),
            ))
    return pd.DataFrame(frames)

def synthetic_frame(stat_group, n=140, rng_seed=0):
    """Deterministic, instant demo series. Used directly in FAST_MODE, or
    as a fallback when the live API returns nothing."""
    rng = np.random.default_rng(rng_seed)
    dates = pd.date_range("2020-03-01", periods=n, freq="D")
    trend = np.linspace(0.0, 1.0, n)
    if stat_group == "hitting":
        lev = 0.260 + 0.05 * trend
        avg = np.clip(lev + rng.normal(0, 0.03, n).cumsum() * 0.004, 0.1, 0.45)
        return pd.DataFrame(dict(
            date=dates, is_home=rng.integers(0, 2, n).astype(int),
            AB=rng.integers(1, 5, n), BB=rng.integers(0, 2, n),
            SF=rng.integers(0, 1, n), HBP=rng.integers(0, 1, n),
            K=rng.integers(0, 3, n),
            s1B=rng.integers(0, 2, n), s2B=rng.integers(0, 1, n),
            s3B=rng.integers(0, 1, n), HR=rng.integers(0, 1, n),
            avg=avg, obp=avg + 0.06, slg=avg + 0.12, ops=avg + 0.18,
        ))
    lev = 3.6 - 0.4 * trend
    era = np.clip(lev + rng.normal(0, 0.4, n).cumsum() * 0.1, 0.5, 9.0)
    return pd.DataFrame(dict(
        date=dates, is_home=rng.integers(0, 2, n).astype(int),
        IP=rng.uniform(5, 8, n), HR=rng.integers(0, 2, n),
        BB=rng.integers(0, 3, n), HBP=rng.integers(0, 2, n),
        K=rng.integers(2, 9, n), ER=rng.integers(0, 4, n),
        era=era, whip=1.1 + 0.3 * np.random.default_rng(8).random(n),
    ))

print("Ingestion + cache + synthetic fallback ready. FAST_MODE =", FAST_MODE)


Ingestion + cache + synthetic fallback ready. FAST_MODE = True


In [6]:
WOBA_W = {"BB": 0.690, "HBP": 0.722, "1B": 0.888, "2B": 1.271, "3B": 1.616, "HR": 2.101}
LEAGUE_STATS = dict(lg_era=4.20, lg_hr=1801.19, lg_bb=1803.29, lg_hbp=1800.39, lg_k=1808.59, lg_ip=1800)
HITTER_ROLLING = 10
PITCHER_ROLLING = 10

def fip_constant(ls):
    return ls["lg_era"] - (13 * ls["lg_hr"] + 3 * (ls["lg_bb"] + ls["lg_hbp"]) - 2 * ls["lg_k"]) / ls["lg_ip"]

def rolling_fip(df, fc, window=PITCHER_ROLLING):
    hbp = df["HBP"].fillna(0.0) if "HBP" in df.columns else pd.Series(0.0, index=df.index)
    num = (13 * df["HR"].rolling(window, 1).sum() + 3 * (df["BB"].rolling(window, 1).sum() + hbp.rolling(window, 1).sum())
           - 2 * df["K"].rolling(window, 1).sum())
    ip = df["IP"].rolling(window, 1).sum().replace(0, np.nan)
    return num / ip + fc

def rolling_woba(df, window=20):
    need = ["s1B", "s2B", "s3B", "HR", "BB", "HBP", "SF"]
    if not all(c in df.columns for c in need):
        return pd.Series(np.nan, index=df.index)
    num = (WOBA_W["BB"] * df["BB"].rolling(window, 1).sum()
           + WOBA_W["HBP"] * df["HBP"].rolling(window, 1).sum()
           + WOBA_W["1B"] * df["s1B"].rolling(window, 1).sum()
           + WOBA_W["2B"] * df["s2B"].rolling(window, 1).sum()
           + WOBA_W["3B"] * df["s3B"].rolling(window, 1).sum()
           + WOBA_W["HR"] * df["HR"].rolling(window, 1).sum())
    den = (df["AB"] + df["BB"] + df["SF"] + df["HBP"]).rolling(window, 1).sum()
    return num / den.replace(0, np.nan)

def build_player_frame(rep, seasons=None):
    """Returns df, feature_cols, target_col, meta for one representative."""
    grp, pos, sgrp, name, pid = rep

    if FAST_MODE or pid is None:
        df = synthetic_frame(sgrp, n=140, rng_seed=abs(hash(name)) % 232)
        used_synthetic = True
    else:
        seasons = seasons or SEASONS_FAST
        fetch = fetch_pitcher_gl if sgrp == "pitching" else fetch_batter_gl
        df = fetch(pid, list(seasons))
        used_synthetic = len(df) < 60
        if used_synthetic:
            df = synthetic_frame(sgrp, n=200, rng_seed=abs(hash(name)) % 232)

    df = df.sort_values("date").reset_index(drop=True)
    df["appearance_num"] = np.arange(len(df))
    df["is_home_bin"] = df["is_home"].astype(int)
    df["rest_days"] = pd.to_datetime(df["date"]).diff().dt.days.fillna(2).clip(upper=10)

    if sgrp == "pitching":
        df["rolling_target"] = rolling_fip(df, fip_constant(LEAGUE_STATS))
    else:
        w = rolling_woba(df)
        df["rolling_target"] = df["ops"].rolling(HITTER_ROLLING, min_periods=1).mean().where(w.isna(), w)

    df["momentum3"] = df["rolling_target"].rolling(3, min_periods=1).mean()
    feat = ["appearance_num", "momentum3", "is_home_bin", "rest_days"]
    df = df.dropna(subset=["rolling_target"]).reset_index(drop=True)
    return df, feat, "rolling_target", dict(name=name, group=grp, stat=sgrp, synthetic=used_synthetic)

print("Aggregation feature builder ready.")


Aggregation feature builder ready.


# 6–7. Walk-Forward Validation & Model Zoo

The original notebook scored a single 80/20 chronological holdout. A single
split can be lucky; the dashboard's `regression.py` blend weights were also
hard-coded. This v2:

1. **TimeSeriesSplit** (5-fold, no-shuffle) — the model is scored on folds it has
   *not* seen during training, exactly like a season rolling forward.
2. **Model zoo** — Ridge (the repo's original v1 baseline) vs SVR, Huber, GPR,
   RandomForest, HistGradientBoosting, and the repo's blended ensemble. The best
   fit on the data is chosen by mean walk-forward score, not by assumption.


In [7]:
WALK_FORWARD_FOLDS = 3     # was 5 -> fewer refits per model
RUNTIME_CAP_ROWS = 60      # was 120 -> half the rows feeding CV
ZOO_MODE = "fast"          # "fast" = Ridge+HistGB only, "full" = adds SVR+Ensemble

def make_ensemble(weights=(0.45, 0.35, 0.20)):
    """Repo regression.py blend: SVR + Huber + GPR (kept for reference/full mode)."""
    svr = SVR(kernel="rbf", C=1.0, gamma="scale")
    huber = HuberRegressor(alpha=1.0)
    gpr = GaussianProcessRegressor(kernel=RBF(1.0) + WhiteKernel(0.1), normalize_y=True, random_state=42)
    return VotingRegressor([("svr", svr), ("huber", huber), ("gpr", gpr)], weights=list(weights))

def make_models(mode=ZOO_MODE):
    models = {
        "Ridge": Pipeline([("s", StandardScaler()), ("m", Ridge(alpha=1.0))]),
        "HistGB": Pipeline([("s", StandardScaler()),
                             ("m", HistGradientBoostingRegressor(random_state=42, max_iter=150,
                                                                 learning_rate=0.08, max_depth=4))]),
    }
    if mode == "full":
        models["SVR"] = Pipeline([("s", StandardScaler()),
                                   ("m", SVR(kernel="rbf", C=5.0, gamma="scale", epsilon=0.05))])
        models["Ensemble"] = Pipeline([("s", StandardScaler()), ("m", make_ensemble())])
    return models

MODELS = make_models()

def ts_scores(estimator, X, y, nsplits=WALK_FORWARD_FOLDS):
    """Walk-forward TimeSeriesSplit evaluation, scaled inside each fold."""
    cv = TimeSeriesSplit(n_splits=nsplits)
    out = cross_validate(estimator, X, y, cv=cv, return_train_score=False,
                          scoring={"r2": "r2", "rmse": "neg_mean_squared_error", "mae": "neg_mean_absolute_error"})
    return dict(r2=float(out["test_r2"].mean()),
                rmse=float(np.sqrt(-out["test_rmse"].mean())),
                mae=float(-out["test_mae"].mean()))

def prepare_panel(maxrows=RUNTIME_CAP_ROWS):
    panels = {}
    for rep in REPRESENTATIVES:
        grp = rep[0]
        df, feat, tcol, meta = build_player_frame(rep)
        if len(df) > maxrows:
            df = df.tail(maxrows).reset_index(drop=True)   # keep walk-forward CV tractable
        X = df[feat].to_numpy(dtype=float)
        y = df[tcol].to_numpy(dtype=float)
        panels[grp] = dict(meta=meta, X=X, y=y, df=df, feat=feat)
    return panels

print("Validation harness + smaller model zoo ready. ZOO_MODE =", ZOO_MODE)


Validation harness + smaller model zoo ready. ZOO_MODE = fast


In [8]:
PANEL = prepare_panel(maxrows=RUNTIME_CAP_ROWS)
print("Panel prepared:", {g: (p["meta"]["name"], p["X"].shape[0]) for g, p in PANEL.items()})

zoo_rows = []
for group, p in PANEL.items():
    for mname, model in MODELS.items():
        s = ts_scores(model, p["X"], p["y"])
        zoo_rows.append(dict(Group=group, Player=p["meta"]["name"], Model=mname, **s))
zoo = pd.DataFrame(zoo_rows)
zoo = zoo.sort_values(["Group", "r2", "rmse"], ascending=[True, False, True]).reset_index(drop=True)
zoo.round(4)


Panel prepared: {'Battery': ('Austin Wells', 60), 'Infield': ('Jazz Chisholm', 60), 'Outfield': ('Cody Bellinger', 60), 'Non-Fielders': ('Giancarlo Stanton', 60)}


,Group,Player,Model,r2,rmse,mae
0,Battery,Austin Wells,Ridge,-0.0514,0.0130,0.0111
1,Battery,Austin Wells,HistGB,-5.4283,0.0318,0.0263
2,Infield,Jazz Chisholm,Ridge,0.0019,0.0280,0.0222
3,Infield,Jazz Chisholm,HistGB,-1.5125,0.0477,0.0370
4,Non-Fielders,Giancarlo Stanton,Ridge,0.1149,0.0139,0.0109
5,Non-Fielders,Giancarlo Stanton,HistGB,-0.7719,0.0232,0.0175
6,Outfield,Cody Bellinger,Ridge,0.3631,0.0101,0.0082
7,Outfield,Cody Bellinger,HistGB,-4.6459,0.0319,0.0262


# Interpreting the zoo table

For each position group's representative, the model with the **highest mean R²**
(and, tied, the lowest RMSE) is the empirical best fit on that player's rolling
data. The table also reveals *how* overfit/the fixed blend is: if
"Ensemble (SVR+Huber+GPR)" does **not** win, the hard-coded weights are worse
than an off-the-shelf alternative for that player — the motivation for the
`GridSearchCV` tuning in the next section.


# 8. Grid-Search Tuning (small, fast)

Only the two zoo models get tuned: Ridge (regularization strength) and
HistGB (learning rate / depth / iterations). SVR/Ensemble tuning is only
added when `ZOO_MODE = "full"`, since those are the slow branches.


In [9]:
def fit_best_per_group(panels, cv=None):
    """Returns {group: (model_name, fitted_pipeline, cv_r2)}."""
    cv = cv or TimeSeriesSplit(n_splits=WALK_FORWARD_FOLDS)
    out = {}
    for group, p in panels.items():
        X, y = p["X"], p["y"]

        candidates = {
            "Ridge": (Pipeline([("s", StandardScaler()), ("m", Ridge())]),
                      {"m__alpha": [0.1, 1.0, 5.0]}),
            "HistGB": (Pipeline([("s", StandardScaler()), ("m", HistGradientBoostingRegressor(random_state=42))]),
                       {"m__learning_rate": [0.05, 0.1], "m__max_iter": [100, 200], "m__max_depth": [3, 5]}),
        }
        if ZOO_MODE == "full":
            candidates["SVR"] = (Pipeline([("s", StandardScaler()), ("m", SVR(kernel="rbf", gamma="scale"))]),
                                  {"m__C": [1.0, 5.0], "m__epsilon": [0.01, 0.05]})

        best_name, best_score, best_est = None, float("-inf"), None
        for name, (pipe, grid) in candidates.items():
            gs = GridSearchCV(pipe, grid, cv=cv, scoring="r2", n_jobs=-1, refit=True)
            gs.fit(X, y)
            if gs.best_score_ > best_score:
                best_name, best_score, best_est = name, float(gs.best_score_), gs.best_estimator_
        out[group] = (best_name, best_est, best_score)
    return out

BEST = fit_best_per_group(PANEL)
for g, (name, est, score) in BEST.items():
    pname = PANEL[g]["meta"]["name"]
    print(f"{g:13s} best={name:8s} cv_r2={score:.4f} player={pname}")


Battery       best=Ridge    cv_r2=-0.0514 player=Austin Wells
Infield       best=Ridge    cv_r2=0.7014 player=Jazz Chisholm
Outfield      best=Ridge    cv_r2=0.3631 player=Cody Bellinger
Non-Fielders  best=Ridge    cv_r2=0.2131 player=Giancarlo Stanton


# 9. Best-Model Trajectories with 95% CI

For each position group we re-fit the tuned winner on the **first 80%** of
appearances (chronological holdout) and overlay the model's trajectory + a 95%
band on the player's real rolling target — the same visual contract the
dashboard's `trajectories.py` renders for its default ensemble.


In [10]:

from sklearn.base import clone


def chrono_fit(p, estimator, bounds=(0.0, 10.0)):
    """Fit tuned estimator on first 80% of appearances; predict all. Return
    per-row predictions, holdout R²/RMSE, and a residual-based 95% CI width."""
    X, y = p['X'], p['y']
    split = max(2, int(len(y) * 0.8))
    est = clone(estimator).fit(X[:split], y[:split])
    y_pred = est.predict(X)
    if bounds is not None:
        y_pred = np.clip(y_pred, *bounds)
    r2h = rmsh = float('nan')
    if len(y) - split >= 2:
        ye = est.predict(X[split:])
        r2h = float(r2_score(y[split:], ye))
        rmsh = float(np.sqrt(mean_squared_error(y[split:], ye)))
    resid_std = float(np.std(y[split:] - est.predict(X[split:]))) if len(y) - split >= 2 else 0.05
    return dict(y_pred=y_pred, r2=r2h, rmse=rmsh, resid_std=resid_std, split=split)


def chart(group):
    p = PANEL[group]
    best_name, best_est, _ = BEST[group]
    fit = chrono_fit(p, best_est)
    x = np.arange(len(p['y']))
    up = fit['y_pred'] + 1.96 * fit['resid_std']
    lo = fit['y_pred'] - 1.96 * fit['resid_std']

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=p['y'], mode='lines', name='Actual (rolling)',
                             line=dict(color='#002D62')))
    fig.add_trace(go.Scatter(x=x, y=fit['y_pred'], mode='lines',
                             name=f'Fit ({best_name})', line=dict(color='#FF6B35', dash='dash')))
    fig.add_trace(go.Scatter(x=np.concatenate([x, x[::-1]]),
                             y=np.concatenate([up, lo[::-1]]), fill='toself',
                             mode='lines', line=dict(width=0),
                             fillcolor='rgba(44,160,44,0.18)', hoverinfo='skip',
                             name='95% band'))
    fig.add_vline(x=fit['split'], line_dash='dash', line_color='black')
    fig.update_layout(
        title=f"{p['meta']['name']} | {group} | {best_name} "
              f"(holdout R²={fit['r2']:.3f}, RMSE={fit['rmse']:.4f})",
        template='plotly_white', xaxis_title='Appearance #', hovermode='x unified')
    return fig, fit


figs = {g: chart(g) for g in PANEL}
for g, (fig, _) in figs.items():
    fig.show()


In [11]:

summary_rows = []
for group, p in PANEL.items():
    best_name, est, r2 = BEST[group]
    fit = chrono_fit(p, est)
    summary_rows.append(dict(Group=group, Player=p['meta']['name'],
                             TunedPick=best_name, cv_r2=r2,
                             HoldoutR2=fit['r2'], HoldoutRMSE=fit['rmse']))
summary = pd.DataFrame(summary_rows)
summary.round(4)


,Group,Player,TunedPick,cv_r2,HoldoutR2,HoldoutRMSE
0,Battery,Austin Wells,Ridge,-0.0514,0.3505,0.0148
1,Infield,Jazz Chisholm,Ridge,0.7014,0.7179,0.0132
2,Outfield,Cody Bellinger,Ridge,0.3631,0.3096,0.0088
3,Non-Fielders,Giancarlo Stanton,Ridge,0.2131,0.7457,0.0079


# 10. Plain-English Picture (non-technical summary)

The "better fit" from the previous sections disappears into jargon (R², RMSE).
This section re-draws the same results as everyday charts — one player per
position group, the model the computer picked for each, and (in plain words)
how reliable that pick is. Everything below is a Plotly figure you can point at
and explain, no stats background needed.


In [12]:

TREND_COLOR = "#FF6B35"
GROUP_COLORS = {"Battery": "#1F77B4", "Infield": "#2CA02C",
                "Outfield": "#FF7F0E", "Non-Fielders": "#9467BD"}

# Plain-language summary per group: what we modelled + the tuned winner.
def plain_narrative(grp):
    return (f"{PANEL[grp]['meta']['name']} ({grp.lower()}) — "
            f"modelled {'pitching (Battery)' if PANEL[grp]['meta']['stat']=='pitching' else 'hitting'} "
            f"rolling series. Best pick: {BEST[grp][0]}.")

# A simple, readable bar: each player's *average* rolling metric level, coloured
# by position group. Bigger bar = that player's typical performance was better.
kpi_rows = []
for grp, p in PANEL.items():
    level = float(np.nanmean(p['y']))
    kpi_rows.append(dict(Group=grp, Player=p['meta']['name'], AvgRollTarget=round(level, 4)))
kpi = pd.DataFrame(kpi_rows)

# --- write the plain-English winner cards as text too ---
for g in PANEL:
    print(plain_narrative(g))

fig_kpi = go.Figure()
for g in PANEL:
    row = kpi[kpi["Group"] == g].iloc[0]
    fig_kpi.add_trace(go.Bar(
        x=[row["Group"]], y=[row["AvgRollTarget"]],
        marker=dict(color=GROUP_COLORS[g]),
        name=g,
        hovertemplate=f"{row['Player']}<br>typical {row['AvgRollTarget']}<extra></extra>"))
fig_kpi.update_layout(
    title="Typical rolling level by position group (your forecast yardstick)",
    yaxis_title="Average rolling metric value",
    xaxis_title="Position group (one rep each)",
    template="plotly_white")
fig_kpi.show()


Austin Wells (battery) — modelled hitting rolling series. Best pick: Ridge.
Jazz Chisholm (infield) — modelled hitting rolling series. Best pick: Ridge.
Cody Bellinger (outfield) — modelled hitting rolling series. Best pick: Ridge.
Giancarlo Stanton (non-fielders) — modelled hitting rolling series. Best pick: Ridge.


In [13]:

# "How much could the forecast be off?" — lower bar = tighter/more reliable
# forecast. Uses the walk-forward RMSE we already computed per group/model.
fig_off = go.Figure()
for i, grp in enumerate(PANEL):
    best_rmse = zoo[(zoo["Group"] == grp) & (zoo["Model"] == BEST[grp][0])]["rmse"]
    best_rm = float(best_rmse.min()) if not best_rmse.empty else float("nan")
    fig_off.add_trace(go.Bar(
        x=[grp], y=[best_rm], name=grp,
        marker=dict(color=GROUP_COLORS[grp]),
        hovertemplate=(f"{PANEL[grp]['meta']['name']}<br>"
                       f"winner: {BEST[grp][0]}<br>typical miss: {best_rm:.4f}<extra></extra>")))
fig_off.update_layout(
    title="Forecast reliability (lower = tighter fit)",
    yaxis_title="Typical forecast error (RMSE)",
    xaxis_title="Position group",
    template="plotly_white")
fig_off.show()

In [14]:

from plotly.subplots import make_subplots

fig_dash = make_subplots(rows=2, cols=2,
    subplot_titles=[f"{PANEL[g]['meta']['name']} — {g}" for g in PANEL])
for idx, grp in enumerate(PANEL):
    p = PANEL[grp]
    fit = chrono_fit(p, BEST[grp][1])
    x = np.arange(len(p['y']))
    r, c = (idx // 2) + 1, (idx % 2) + 1
    fig_dash.add_trace(go.Scatter(x=x, y=p['y'], mode="lines", name="Actual",
                                  line=dict(color="#002D62")), row=r, col=c)
    fig_dash.add_trace(go.Scatter(x=x, y=fit['y_pred'], mode="lines",
                                  name="Forecast", line=dict(color="#FF6B35", dash="dash")),
                       row=r, col=c)
    fig_dash.add_vline(x=fit['split'], row=r, col=c, line_dash="dash",
                       line_color="#999")
fig_dash.update_layout(title="Your forecast dashboard — one panel per position group",
                       template="plotly_white", showlegend=False, height=560)
fig_dash.show()


# 11. Wrap-Up: What "better fit" means here

- **Validation gap closed.** The original notebook reported one 80/20 holdout;
  this version scores every model on a no-shuffle **TimeSeriesSplit** and, for
  the final trajectory, re-runs the chronological holdout — the reported R²/RMSE
  are genuinely out-of-sample.
- **Position-aware.** Every representative was routed through the repo's
  `Battery/Infield/Outfield/Non-Fielders` taxonomy and its `pitching` vs
  `hitting` metric set, instead of the old flat Offense/Defense split.
- **Empirical model pick.** The tuned winner (HistGB vs SVR vs the repo ensemble)
  is chosen per group by walk-forward CV, so the incumbents' blend weights are
  validated rather than assumed.

+ the representative players' surnames).


### Below is a test method to filter and fetch news. Currently it is facing offline issues

In [15]:

def fetch_and_filter_news(team_keywords):
    rss_url = "https://www.mlb.com/feeds/news/rss.xml"
    try:
        res = requests.get(rss_url, headers=HEADERS, timeout=15)
        root = ET.fromstring(res.content)
    except Exception as exc:
        print("news skipped (offline/error):", exc)
        return pd.DataFrame(columns=["title", "published", "link"])

    articles = []
    for item in root.findall("./channel/item"):
        title = item.find("title").text if item.find("title") is not None else ""
        link = item.find("link").text if item.find("link") is not None else ""
        pub = item.find("pubDate").text if item.find("pubDate") is not None else ""
        desc = item.find("description").text if item.find("description") is not None else ""
        corpus = f"{title} {desc}".lower()
        if any(kw.lower() in corpus for kw in team_keywords if kw):
            articles.append({"title": title, "published": pub, "link": link})
    return pd.DataFrame(articles)

news_kw = ["Yankees"] + [p["meta"]["name"].split()[-1] for p in PANEL.values()]
df_news = fetch_and_filter_news(news_kw)
print(f"Filtered {len(df_news)} headlines for {news_kw}")
df_news.head(5)


news skipped (offline/error): HTTPSConnectionPool(host='www.mlb.com', port=443): Max retries exceeded with url: /feeds/news/rss.xml (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f684b3f3890>: Failed to resolve 'www.mlb.com' ([Errno -3] Temporary failure in name resolution)"))
Filtered 0 headlines for ['Yankees', 'Wells', 'Chisholm', 'Bellinger', 'Stanton']


,title,published,link
